In [2]:
import os
import re
import datetime

# 3. Sensitive keywords to check in filenames
KEYWORDS = ["password", "secret", "credential", "apikey", "api_key", "token"]

# 4. Regular expression patterns to identify patterns like email addresses inside text files
PATTERNS = {
    "Email Address": r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b',
    "API Key / Token": r'\b(A3T[A-Z0-9]|AKIA|AGPA|AIDA|AROA|AIPA|ANPA|ANVA|ASIA)[A-Z0-9]{16}\b',
    "Generic Secret Assignment": r'(?i)(password|secret|api[_-]?key)\s*[:=]\s*[\'"][^\'"]+[\'"]'
}

def scan_directory(target_dir):
    # 1. Allow user to specify directory and handle path formatting
    target_dir = target_dir.strip().strip('"\'')
    
    if not os.path.exists(target_dir):
        print(f"Error: Directory '{target_dir}' does not exist.")
        return []

    if not os.path.isdir(target_dir):
        print(f"Error: '{target_dir}' is a file, not a directory.")
        return []

    print(f"\n[*] Scanning directory: {os.path.abspath(target_dir)}")
    findings = []

    # 2. Use the os library (os.walk) to navigate through directories and files
    for root, dirs, files in os.walk(target_dir):
        for file in files:
            file_path = os.path.join(root, file)
            file_issues = []

            # 3. Identify filenames containing keywords (Password, Secret, etc.)
            lower_filename = file.lower()
            for keyword in KEYWORDS:
                if keyword in lower_filename:
                    file_issues.append(f"Filename contains restricted keyword: '{keyword}'")

            # 4. Read text files and use regular expressions for patterns (like email addresses)
            if file.endswith(('.txt', '.env', '.json', '.log', '.py', '.config', '.md')):
                try:
                    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                        for line_num, line in enumerate(f, 1):
                            for pattern_name, regex in PATTERNS.items():
                                if re.search(regex, line):
                                    file_issues.append(f"Found {pattern_name} at line {line_num}")
                except Exception as e:
                    print(f"Could not read file {file_path}: {e}")

            # 5 & 6. Identify potentially exposed information and store locations
            if file_issues:
                findings.append({
                    "path": file_path,
                    "issues": file_issues
                })

    return findings

def generate_report(target_dir, findings):
    # 7. Generate and save a scan report
    report_filename = "scan_report.txt"
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    report_lines = [
        "=" * 65,
        " SENSITIVE DATA & INFORMATION LEAKAGE SCAN REPORT",
        "=" * 65,
        f"Scan Timestamp   : {timestamp}",
        f"Target Directory : {os.path.abspath(target_dir)}",
        f"Total Flagged    : {len(findings)} file(s)",
        "=" * 65 + "\n"
    ]

    if not findings:
        report_lines.append("Status: CLEAN. No sensitive files or data leaks detected.\n")
    else:
        for idx, item in enumerate(findings, 1):
            report_lines.append(f"[{idx}] File Location: {item['path']}")
            for issue in item['issues']:
                report_lines.append(f"    - {issue}")
            report_lines.append("-" * 65)

    report_text = "\n".join(report_lines)

    # Save report to a text file
    with open(report_filename, 'w', encoding='utf-8') as report_file:
        report_file.write(report_text)

    print("\n" + report_text)
    print(f"\n[*] Scan report successfully saved to: {os.path.abspath(report_filename)}")

if __name__ == "__main__":
    # User Interaction via Terminal
    user_input = input("Enter the directory path to scan: ")
    scan_results = scan_directory(user_input)
    if scan_results or os.path.exists(user_input.strip().strip('"\'')):
        generate_report(user_input, scan_results)

Enter the directory path to scan:  E:\Doc



[*] Scanning directory: E:\Doc

 SENSITIVE DATA & INFORMATION LEAKAGE SCAN REPORT
Scan Timestamp   : 2026-11-16 17:26:52
Target Directory : E:\Doc
Total Flagged    : 1 file(s)

[1] File Location: E:\Doc\file.txt
    - Found Generic Secret Assignment at line 3
    - Found API Key / Token at line 4
    - Found Email Address at line 5
-----------------------------------------------------------------

[*] Scan report successfully saved to: C:\Users\HP\scan_report.txt
